In [1]:
import os
import json
import pandas as pd
from sentence_transformers import SentenceTransformer
!pip install faiss-cpu
import faiss
import numpy as np
import textwrap
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Torch:", torch.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 77.3 MB/s eta 0:00:00
Torch: 2.8.0+cu126


In [2]:
KB_DIR = "/content/drive/MyDrive/Module3"
os.makedirs(KB_DIR, exist_ok=True)

kb_files = [os.path.join(KB_DIR, f) for f in os.listdir(KB_DIR) if f.endswith(".txt")]

if not kb_files:
    print("⚠ No KB files found, creating sample knowledge base...")

    sample_text = """
    Wheat requires moderate rainfall between 50–100cm and grows well in loamy soil.
    Tomatoes are susceptible to early blight and require good drainage and regular nitrogen fertilizers.
    Banana grows best in warm climates above 20°C with high humidity.
    Cotton needs high temperature and black soil for optimal yield.
    Rice needs standing water, clay soil, and high humidity above 70%.
    """

    with open(os.path.join(KB_DIR, "sample_kb.txt"), "w") as f:
        f.write(sample_text)

    kb_files = [os.path.join(KB_DIR, "sample_kb.txt")]

kb_files


['/content/drive/MyDrive/Module3/sample_kb.txt']

In [3]:
kb_texts = []

for file in kb_files:
    with open(file, "r", encoding="utf-8") as f:
        kb_texts.append(f.read())

full_kb = "\n".join(kb_texts)
len(full_kb), full_kb[:300]


(403,
 '\n    Wheat requires moderate rainfall between 50–100cm and grows well in loamy soil.\n    Tomatoes are susceptible to early blight and require good drainage and regular nitrogen fertilizers.\n    Banana grows best in warm climates above 20°C with high humidity.\n    Cotton needs high temperature and bl')

In [4]:
def chunk_text(text, chunk_size=300, overlap=80):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(full_kb)
len(chunks)


1

In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(chunks, convert_to_numpy=True)
embeddings.shape


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(1, 384)

In [6]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print("FAISS DB size:", index.ntotal)


FAISS DB size: 1


In [8]:
FAISS_PATH = "/content/drive/MyDrive/data_now/faiss_index.bin"
META_PATH  = "/content/drive/MyDrive/data_now/chunks.json"

faiss.write_index(index, FAISS_PATH)

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump({"chunks": chunks}, f)

print("Saved FAISS index and chunks.")


Saved FAISS index and chunks.


In [9]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
def retrieve(query, k=3):
    q_emb = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(q_emb, k)
    return [chunks[i] for i in indices[0]]


In [11]:
def generate_answer(query):
    retrieved = retrieve(query, k=3)
    context = "\n".join(retrieved)

    prompt = f"""You are an agriculture expert.
Use the following context to answer the question.

Context:
{context}

Question: {query}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = generator_model.generate(**inputs, max_new_tokens=200)

    return tokenizer.decode(outputs[0], skip_special_tokens=True), retrieved


In [12]:
query = "Which crop requires clay soil and high humidity?"
answer, ctx = generate_answer(query)

print("Answer:", answer)
print("\nRetrieved Context:", ctx)


Answer: Rice

Retrieved Context: ['Wheat requires moderate rainfall between 50–100cm and grows well in loamy soil. Tomatoes are susceptible to early blight and require good drainage and regular nitrogen fertilizers. Banana grows best in warm climates above 20°C with high humidity. Cotton needs high temperature and black soil for optimal yield. Rice needs standing water, clay soil, and high humidity above 70%.', 'Wheat requires moderate rainfall between 50–100cm and grows well in loamy soil. Tomatoes are susceptible to early blight and require good drainage and regular nitrogen fertilizers. Banana grows best in warm climates above 20°C with high humidity. Cotton needs high temperature and black soil for optimal yield. Rice needs standing water, clay soil, and high humidity above 70%.', 'Wheat requires moderate rainfall between 50–100cm and grows well in loamy soil. Tomatoes are susceptible to early blight and require good drainage and regular nitrogen fertilizers. Banana grows best in w

In [15]:
%%writefile src/rag_qa_tool.py
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class RAGQA:
    def __init__(self, index_path="/content/drive/MyDrive/data_now/faiss_index.bin",
                       meta_path="/content/drive/MyDrive/data_now/chunks.json"):
        self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.index = faiss.read_index(index_path)

        with open(meta_path, "r", encoding="utf-8") as f:
            self.chunks = json.load(f)["chunks"]

        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

    def retrieve(self, query, k=3):
        q_emb = self.model.encode([query], convert_to_numpy=True)
        _, indices = self.index.search(q_emb, k)
        return [self.chunks[i] for i in indices[0]]

    def answer(self, query):
        context = "\n".join(self.retrieve(query))
        prompt = f"""You are an agriculture expert.
        Use this context to answer.

        Context:
        {context}

        Question: {query}
        Answer:
        """
        inputs = self.tokenizer(prompt, return_tensors="pt")
        output = self.gen_model.generate(**inputs, max_new_tokens=200)
        ans = self.tokenizer.decode(output[0], skip_special_tokens=True)
        return ans


Writing src/rag_qa_tool.py


In [14]:
import os
os.makedirs("src", exist_ok=True)


In [1]:
import tensorflow as tf

try:
    model = tf.keras.models.load_model(r"D:\Final Project\Buildable-ML-DL-Fellowship\Final_project\notebooks\models\disease_cnn.h5")
    print("Model loaded OK")
except Exception as e:
    print("Error:", e)


Error: Unable to synchronously open file (bad object header version number)
